# 📘 Annotated edition · 06_answer_cold_run

A copy of `notebooks/06_answer_cold_run.ipynb`. Original cells are unchanged (this notebook has no saved outputs: it was never run). New **📘 Explanation** and **✅ Answer** cells are added.

**The point of this notebook:** it's the one the panel asks you to run **from a fresh Codespace** ("cold run"). It reads the clean tables, computes **the one headline number** with its numerator, denominator, period and comparison, writes it to Excel next to its sentence, and prints it in the last cell. It's short and boring on purpose: nothing clever, so nothing can go wrong in front of the panel.

**What it prints when run on the current clean data** (recomputed): `number: 5.0` (452,933 / 431,261 tests). With the 30 June fix it's still **+5.0%** (455,383 / 433,611), so the rounded headline is the same and the levels are right.

> ℹ️ **One line changed in this annotated copy:** `ROOT = ... if pathlib.Path.cwd().name in ("notebooks", "notebook_new") else ...`. The original only recognised a folder called `notebooks`. Run from `notebook_new/`, it would have looked for `notebook_new/data/raw` and failed. Nothing else in the code was changed.

# 06 · The answer, the board number, and the cold run

**Produces:** `slides/board_number.xlsx` (the one number with its denominator, period and comparison beside it), the headline printed in the last cell, and a cold-run row in the decisions log.

**Needs:** notebooks 03 and 05 run.

This notebook is short on purpose. It is the one the panel will ask you to run from a fresh Codespace, so it reads clean tables and does nothing clever.

#### 📘 Explanation: header

- **`board_number.xlsx`**: the board works in Excel, so the number goes there *with* its sentence, denominator, period and caveat beside it, plus a formula that recomputes it.
- **The headline printed in the last cell**: the number on slide 3 must be **exactly** this printed number. If they differ, one of them is wrong.
- **A cold-run row in the decisions log**: evidence that someone other than the author reproduced the number from scratch.

## CONFIG

#### 📘 Explanation: the CONFIG cell, line by line

This cell is the same in notebooks 01–06. Explained once in full:

| Code | What it does and why |
|---|---|
| `import sys, pathlib, time` | `sys` to edit the import path, `pathlib` for file paths that work on Windows and Linux, `time` to time steps. |
| `ROOT = pathlib.Path.cwd().parent if ... == "notebooks" else pathlib.Path.cwd()` | Finds the **project root**. Jupyter usually starts inside the notebook's folder, so the root is one level up. When run from the root itself (for example a script), `cwd()` is already the root. Every path below is built from `ROOT`, so the notebook works on any machine. |
| `sys.path.insert(0, str(ROOT / "src"))` | Puts `src/` at the front of Python's import search path, so `from kalpa import ...` finds the project's own package without `pip install`. |
| `import numpy as np` / `import pandas as pd` | The two core libraries: arrays and fast maths / tables. |
| `from kalpa import loaders, models, checks, logs` | The shared project package: **loaders** read the raw files, **models** are pydantic "contracts" for a valid record, **checks** print PASS/FAIL, **logs** append rows to the markdown logs in `docs/`. |
| `pd.set_option("display.max_columns", 40)` / `("display.width", 160)` | Print wide tables in full instead of truncating them with `...`. |
| `CONFIG = {...}` | All run-specific settings in one place (see notebook 00): brief, group, `raw` and `clean` folders as `pathlib` paths, the two half-years being compared, the seed, and `workers` (`None` means use every CPU core in the process pool). |
| `CONFIG["clean"].mkdir(parents=True, exist_ok=True)` | Creates `data/clean/` if it doesn't exist, and doesn't fail if it does. A fresh clone can run straight away. |
| `rng = np.random.default_rng(CONFIG["seed"])` | A seeded random generator **object**. Anything random (the shuffle test in 05) uses `rng`, so results are reproducible. |
| `print(pd.__version__, np.__version__)` | Prints library versions (**2.3.3** and **2.3.5** in the saved run). If a re-run elsewhere gives different numbers, a version difference is the first suspect. pandas 3.x, for example, changed the default string dtype. |

In [ ]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name in ("notebooks", "notebook_new") else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "<LETTER>",
    "group": "<N>",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

#### 📘 Explanation: computing the headline

- Reads `bookings_clean.csv.gz` (the output of 03).
- `h = {"last": bk[between(period_last)], "this": bk[between(period_this)]}`: the two half-years. ⚠️ The same 30 June issue as notebook 05. Use the half-open interval (`>= start` and `< end + 1 day`) so the levels match SQL.
- `tests_last`, `tests_this`: total tests (sum of `n_tests`) in each half, as plain Python `int`s (openpyxl and JSON can't handle NumPy integers well).
- `growth = (tests_this / tests_last - 1) * 100`, the % change.
- `HEADLINE = {...}`: the number **with everything a board member needs to interpret it**:
  - `claim`: the one sentence (still a placeholder `<...>`, see the answer below);
  - `number`: rounded to 1 decimal. More precision than that is false precision;
  - `numerator` / `denominator`: the two counts, so anyone can redo the division;
  - `unit`: exactly what is counted. ⚠️ It says "tests on bookings that were collected or no-show", but the code **doesn't filter by status** (cancelled bookings are included). Fix one or the other: with the filter, growth is **+4.2%** (438,344 / 420,658, full days); without it, +5.0%;
  - `period`, `comparison` (the plan), `caveat`.
- The loop prints every field, right-aligned.

In [ ]:
bk = pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", parse_dates=["slot_at"], dtype={"booking_id": "string", "person_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
h = {"last": bk[bk["slot_at"].between(*CONFIG["period_last"])], "this": bk[bk["slot_at"].between(*CONFIG["period_this"])]}
tests_last, tests_this = int(h["last"]["n_tests"].sum()), int(h["this"]["n_tests"].sum())
growth = (tests_this / tests_last - 1) * 100
HEADLINE = {"claim": "<one sentence for the board>", "number": round(growth, 1), "numerator": tests_this, "denominator": tests_last,
            "unit": "tests on bookings that were collected or no-show, after the identity rule in docs/decisions-log.md",
            "period": "H1 2026 against H1 2025, six cities", "comparison": "plan: +18 percent", "caveat": "<the one thing that would change the claim>"}
for k, v in HEADLINE.items(): print(f"{k:>12}: {v}")

#### 📘 Reading the output (recomputed. The saved file has none)

```
       claim: <one sentence for the board>
      number: 5.0
   numerator: 452933
 denominator: 431261
        unit: tests on bookings that were collected or no-show, after the identity rule in docs/decisions-log.md
      period: H1 2026 against H1 2025, six cities
  comparison: plan: +18 percent
      caveat: <the one thing that would change the claim>
```
With full-day periods: numerator 455,383, denominator 433,611, still +5.0%. The plan's implied growth is +18.9% (515,740 planned tests vs 433,611 actual in H1 2025), so "plan: +18 percent" is right if written as +19%.

## The board number, in Excel, with its sentence beside it

#### 📘 Explanation: writing the Excel board number with openpyxl

- `Workbook()` creates a workbook. `ws = wb.active` is its first sheet, renamed "Board number".
- `rows`: a list of (label, value) pairs that become rows 1–11:

| Row | A | B |
|---|---|---|
| 1 | Kalpa Health · Brief · Group (title) | |
| 2 | (blank) | |
| 3 | The number | `+5.0%` (as **text**) |
| 4 | What it counts | the unit |
| 5 | Numerator | 452,933 (a number) |
| 6 | Denominator | 431,261 (a number) |
| 7–10 | Period, Compared with, The sentence, The caveat | |
| 11 | Formula check | `=ROUND((B5/B6-1)*100,1)` |

- `ws.append(list(r))` writes each pair as a row.
- Fonts: bold 14 pt title, **24 pt** for the number. Column widths 18 and 90. `wrap_text=True` so long sentences wrap.
- `wb.save(ROOT / "slides" / "board_number.xlsx")`.

**Why the formula in B11:** Excel recomputes the growth **from the numerator and denominator cells** when the file is opened. If B11 (`5.0`) disagrees with B3 (`+5.0%`), someone typed a number by hand or used different counts. It's a self-check built into the deliverable. (openpyxl writes formulas but doesn't calculate them, so the value appears only when Excel opens the file.)

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
wb = Workbook(); ws = wb.active; ws.title = "Board number"
rows = [("Kalpa Health · Brief " + CONFIG["brief"] + " · Group " + CONFIG["group"], ""), ("", ""),
        ("The number", f"{HEADLINE['number']:+.1f}%"), ("What it counts", HEADLINE["unit"]), ("Numerator", HEADLINE["numerator"]), ("Denominator", HEADLINE["denominator"]),
        ("Period", HEADLINE["period"]), ("Compared with", HEADLINE["comparison"]), ("The sentence", HEADLINE["claim"]), ("The caveat", HEADLINE["caveat"]),
        ("Formula check", "=ROUND((B5/B6-1)*100,1)")]
for r in rows: ws.append(list(r))
ws["A1"].font = Font(bold=True, size=14); ws["B3"].font = Font(bold=True, size=24)
ws.column_dimensions["A"].width = 18; ws.column_dimensions["B"].width = 90
for row in ws.iter_rows(min_row=3, max_row=11): row[1].alignment = Alignment(wrap_text=True, vertical="top")
(ROOT / "slides").mkdir(exist_ok=True)
wb.save(ROOT / "slides" / "board_number.xlsx")
print("saved slides/board_number.xlsx; the formula in B11 must agree with B3 when opened")

#### 📘 Reading the output
Prints `saved slides/board_number.xlsx; the formula in B11 must agree with B3 when opened`. Open the file and check that B11 shows **5.0**.

## The cold run

#### 📘 Explanation: the cold run

- `subprocess.check_output(["git", "rev-parse", "--short", "HEAD"])` gets the short **commit hash** of the code that produced the number. It ties the headline to an exact version of the repository. If git isn't available (or this isn't a repo, as in the current folder), it prints "no git".
- Prints `commit <sha>  headline +5.0%  (452,933 / 431,261)`.
- `logs.cold_run(...)` (commented out) appends a row to the "Cold runs" table in `docs/decisions-log.md`: date, who, fresh Codespace or not, runtime, headline, whether it matched the last run, and why not if it didn't. Uncomment it and fill it in **when a teammate does the cold run**.
- Two checks: the number is finite (no division by zero or NaN), and the claim sentence has been written (it doesn't start with `<`).
- `checks.summary()`.

**What "idempotent" means here:** running notebooks 01 → 06 twice gives the same files and the same number. Inputs are fixed, there's no randomness without a seed, and outputs are overwritten, not appended. One exception to fix: the `logs.*` calls in 01 **append** a row on every run.

In [ ]:
import subprocess, sys
try:
    sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
except Exception:
    sha = "no git"
print(f"commit {sha}  headline {HEADLINE['number']:+.1f}%  ({HEADLINE['numerator']:,} / {HEADLINE['denominator']:,})")
# Record it the moment it is run. Uncomment and fill in.
# logs.cold_run(who="<name>", fresh=True, runtime="<m:ss>", headline=f"{HEADLINE['number']:+.1f}%", same_as_last="yes", why="")
checks.check("headline is a finite number", np.isfinite(HEADLINE["number"]))
checks.check("the claim sentence has been written", not HEADLINE["claim"].startswith("<"))
checks.summary()

#### 📘 Reading the output (recomputed)
```
commit no git  headline +5.0%  (452,933 / 431,261)
PASS  headline is a finite number
FAIL  the claim sentence has been written
1 of 2 checks passed
```
The FAIL is deliberate until you write the claim. See the answer below for a sentence and a caveat to paste in.

## The last cell

The number on slide 3 is the one printed above. If it is not, one of the two is wrong, and the panel will ask which.

#### ✅ Answers: the last cell, and the placeholders this notebook asks you to fill

**"The number on slide 3 is the one printed above. If it isn't, which is wrong?"**

Printed: **+5.0%**. Slide 3 must show +5.0% with **455,383 / 433,611** tests (after applying the 30 June fix in this notebook and notebook 05, so the levels match SQL). If the slide says anything else, check in this order: (1) was the slide typed from an older run? Compare the commit hash. (2) Does the slide use a different denominator (collected only, or collected + no-show)? (3) Different period boundaries?

**A claim sentence to fill `HEADLINE["claim"]` (brief A):**
> "Tests grew 5.0% in January–June 2026 over the same months of 2025 (455,383 vs 433,611), well short of the plan's 19%. The growth came from more patients (+3.8%) and more tests per visit (+2.2%), while patients visited slightly less often (−1.0%). Jaipur shrank 10.6% and Chennai was flat."

**A caveat to fill `HEADLINE["caveat"]`:**
> "Chennai and Jaipur moved to a new booking system in April 2026. Counting a rescheduled visit once (our rule) rather than twice lowers growth from 5.8% to 5.0%, and there's no overlap month to calibrate the two systems."

**Make `unit` match the code:** either keep "tests on all bookings after the identity rule" (+5.0%), or filter to collected + no-show and report **+4.2%**. Don't mix the label of one with the number of the other.

**Log the cold run** when a teammate runs it on a fresh Codespace:
`logs.cold_run(who="<teammate>", fresh=True, runtime="<m:ss>", headline="+5.0%", same_as_last="yes", why="")`